In [ ]:
from pathlib import Path
import numpy as np
from astropy.io import fits
from PIL import Image

folder = Path("260308_darkcurrent_gifs/flat_fits/redo")

# MAKE BROWSE IMAGES 

flat_dir = folder / "flat_fits"
browse_dir = folder / "browse"
flat_dir.mkdir(exist_ok=True)
browse_dir.mkdir(exist_ok=True)
for fits_file in folder.glob("*.fits"):
    with fits.open(fits_file) as hdul:
        data = hdul[0].data
    bands, rows, cols = data.shape
    
    with fits.open(dark) as hdul:
        data = hdul[0].data
    dark_flat = data.transpose(1, 0, 2)  
    
    image = tmp

    out_fits = flat_dir / f"{fits_file.stem}_flat.fits"
    fits.writeto(out_fits, image, overwrite=True)

    frames = {
        "fifth": image[5],  
        "fifthfromlast": image[-5]      
    }
    
    for label, browse in frames.items():
        lo, hi = np.percentile(browse, (2, 98))
        stretched = np.clip((browse - lo) / (hi - lo), 0, 1)
        browse_uint8 = (stretched * 255).astype(np.uint8)
        img = Image.fromarray(browse_uint8, mode="L")
        out_jpg = browse_dir / f"{fits_file.stem}_{label}.jpg"
        img.save(out_jpg, "JPEG", quality=95)

print("done!!!")

In [ ]:
# MAKE GIFS FOR EACH FRAME


folder = Path("260308_darkcurrent_gifs/flat_fits/redo")
gif_dir = folder / "gifs"
gif_dir.mkdir(exist_ok=True)

for fits_file in folder.glob("*.fits"):
    print("Processing", fits_file.name)

    with fits.open(fits_file) as hdul:
        data = hdul[0].data 

    # lets use the same stretch for the whole gif 
    lo, hi = np.percentile(data, (2, 98))

    frames = []

    for i in range(data.shape[0]):

        browse = data[i]
        
        stretched = np.clip((browse - lo) / (hi - lo), 0, 1)

        browse_uint8 = (stretched * 255).astype(np.uint8)

        img = Image.fromarray(browse_uint8, mode="L")
        frames.append(img)

    gif_path = gif_dir / f"{fits_file.stem}.gif"

    frames[0].save(
        gif_path,
        save_all=True,
        append_images=frames[1:],
        duration=80,  
        loop=0
    )
